<a href="https://colab.research.google.com/github/asomers205/DS2002FA26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"The total revenue across all orders is ${total_revenue:.2f}.")
print(f"The total number of units sold is {total_units}.")

The total revenue across all orders is $8520.00.
The total number of units sold is 783.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
total_revenue = df['revenue'].sum()
percentage_share = (revenue_by_category / total_revenue * 100).round(2)

by_category = pd.DataFrame({
    'revenue': revenue_by_category,
    'percentage_of_total': percentage_share
})

print("Revenue by Category (highest to lowest):")
print(by_category)
print("\nThis table shows the total revenue generated by each product category, along with its percentage contribution to the overall revenue. Food generates the most revenue, followed by Drink, Merch, and RainGear.")

Revenue by Category (highest to lowest):
          revenue  percentage_of_total
category                              
Food       4293.0                50.39
Merch      1771.5                20.79
Drink      1554.0                18.24
RainGear    901.5                10.58

This table shows the total revenue generated by each product category, along with its percentage contribution to the overall revenue. Food generates the most revenue, followed by Drink, Merch, and RainGear.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vendor_summary = df.groupby('vendor_id').agg(average_revenue=('revenue', 'mean'), order_count=('vendor_id', 'count'))
vendor_summary_sorted = vendor_summary.sort_values(by='average_revenue', ascending=False)

print("Vendor Average Order Revenue and Order Count:")
print(vendor_summary_sorted)

print(f"\nThe vendor with the highest average order revenue is {vendor_summary_sorted.index[0]} with an average of ${vendor_summary_sorted['average_revenue'].iloc[0]:.2f} across {vendor_summary_sorted['order_count'].iloc[0]} orders.")

Vendor Average Order Revenue and Order Count:
           average_revenue  order_count
vendor_id                              
V-01             22.595745           94
V-18             21.750000          108
V-05             20.580645           93
V-10             20.314286          105

The vendor with the highest average order revenue is V-01 with an average of $22.60 across 94 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_revenue = by_category.loc['Merch', 'revenue']
total_revenue = df['revenue'].sum()
merch_percentage = (merch_revenue / total_revenue) * 100
print(merch_percentage)

print(f"Merch revenue accounts for {merch_percentage:.1f}% of the total revenue.")

20.79225352112676
Merch revenue accounts for 20.8% of the total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

# Check if row count changed
initial_row_count = len(df)
merged_row_count = len(joined)
print(f"Initial row count: {initial_row_count}")
print(f"Merged row count: {merged_row_count}")
assert initial_row_count == merged_row_count, 'Row count changed after merge!'

# Check if total revenue changed
initial_total_revenue = df['revenue'].sum()
merged_total_revenue = joined['revenue'].sum()
print(f"Initial total revenue: ${initial_total_revenue:.2f}")
print(f"Merged total revenue: ${merged_total_revenue:.2f}")
assert abs(initial_total_revenue - merged_total_revenue) < 0.01, 'Total revenue changed after merge!'

# Find the unmatched vendor
unmatched_vendor_id = joined[joined['vendor_name'].isna()]['vendor_id'].unique()
print(f"\nThe unmatched vendor ID is: {unmatched_vendor_id[0] if len(unmatched_vendor_id) > 0 else 'None'}")

print("\n--- Joined DataFrame Head ---")
print(joined.head())

Initial row count: 400
Merged row count: 400
Initial total revenue: $8520.00
Merged total revenue: $8520.00

The unmatched vendor ID is: V-18

--- Joined DataFrame Head ---
  vendor_id  category  qty  price  revenue      vendor_name
0      V-10     Drink    2   24.0     48.0  Cav Merch North
1      V-18  RainGear    1   12.0     12.0              NaN
2      V-18     Drink    3    4.5     13.5              NaN
3      V-10      Food    2   12.0     24.0  Cav Merch North
4      V-18     Drink    3    7.5     22.5              NaN


In [7]:
joined['vendor_name'] = joined['vendor_name'].fillna('Unmatched Vendor')
print("Filled NaN vendor names with 'Unmatched Vendor'.")
print("\n--- Joined DataFrame Head after filling NaN ---")
print(joined.head())

Filled NaN vendor names with 'Unmatched Vendor'.

--- Joined DataFrame Head after filling NaN ---
  vendor_id  category  qty  price  revenue       vendor_name
0      V-10     Drink    2   24.0     48.0   Cav Merch North
1      V-18  RainGear    1   12.0     12.0  Unmatched Vendor
2      V-18     Drink    3    4.5     13.5  Unmatched Vendor
3      V-10      Food    2   12.0     24.0   Cav Merch North
4      V-18     Drink    3    7.5     22.5  Unmatched Vendor


### The unmatched vendor, and what I did about it:

The unmatched vendor ID `V-18` was identified because it was present in the `df` DataFrame but not in the provided `vendor_names` lookup table. To address this for the current analysis, I have replaced the `NaN` values in the `vendor_name` column with the placeholder 'Unmatched Vendor'.

Other potential actions could include:
- Updating the `vendor_names` DataFrame with the correct name for `V-18` if it becomes known.
- Investigating the source data to understand why `V-18` is present without a corresponding name.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [8]:
pivot_table = pd.pivot_table(joined,
                             values='revenue',
                             index='vendor_name',
                             columns='category',
                             aggfunc='sum',
                             margins=True) # Add row and column totals

print("Revenue Pivot Table (Vendors vs. Categories with Totals):")
print(pivot_table.round(2))

print("\nThis pivot table shows the revenue generated by each vendor for each product category, with 'All' rows and columns providing the total revenue for each vendor and category, respectively. The 'All' cell in the bottom right indicates the grand total revenue.")

Revenue Pivot Table (Vendors vs. Categories with Totals):
category           Drink    Food   Merch  RainGear     All
vendor_name                                               
Cav Merch North    502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers       171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos      298.5   882.0   489.0     244.5  1914.0
Unmatched Vendor   582.0  1018.5   508.5     240.0  2349.0
All               1554.0  4293.0  1771.5     901.5  8520.0

This pivot table shows the revenue generated by each vendor for each product category, with 'All' rows and columns providing the total revenue for each vendor and category, respectively. The 'All' cell in the bottom right indicates the grand total revenue.


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [9]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

### Q7.a: What would you tell these vendors to do differently next game?



 Based on the analysis, Unmatched Vendor (V-18) generated the highest total revenue of 2349.0 - primarily driven by Food (1018.5) and Drink (582.0) sales, despite not having a known name. Other vendors, such as Hoos Burgers (V-01), achieved the highest average order revenue at $22.60. I would advise other vendors to focus on strategies that increase average order value, potentially by bundling items or promoting higher-priced products. Additionally, they should analyze the product categories that contributed most to the 'Unmatched Vendor's' success to understand potential high-demand areas.


### Q7.b: Which of your seven answers is the least trustworthy, and why?

The answer regarding the `vendor_id` and `vendor_name` analysis (Q5, Q6) is the least trustworthy. The presence of an 'Unmatched Vendor' (V-18) significantly impacts the accuracy of vendor-specific reporting, especially in the pivot table (Q6). Since we lack specific information for this vendor, any conclusions drawn about individual vendor performance are skewed. For instance, the 'Unmatched Vendor' appears to be the highest revenue generator, but without knowing its true identity, we cannot provide actionable insights or compare its performance accurately with named vendors.